# AGH DS Laboratory 6 - Actor Model with Ray Framework

## Introduction

Ray is a general-purpose framework for programming a cluster made by UC Berkeley's RISELab. It enables developers to easily parallelize their Python applications or build new ones, and run them at any scale, from a laptop to a large cluster. It also provides a highly flexible, yet minimalist and easy to use API. 

#### Documentation Reference Links:

Ray documentation: https://docs.ray.io/en/latest/ \
Ray GitHub: https://github.com/ray-project/ray \
Ray Core: https://docs.ray.io/en/latest/ray-core/walkthrough.html \
Ray Client: https://docs.ray.io/en/latest/cluster/running-applications/job-submission/ray-client.html \

### Środowisko Python

**`ModuleNotFoundError: No module named 'ray'`** — wybrany kernel to lokalny Python **bez** Ray.

Wybierz jedną opcję:

1. **Docker (zalecane):** `docker compose up -d` w `for-students`, potem http://localhost:8888 (token `raylab`).
2. **Lokalnie w Cursor/VS Code:** w terminalu  
   `pip install -r ../requirements.txt`  
   i wybierz ten sam interpreter jako kernel Jupyter.


***
## Part 1 - Remote Functions

This script is too slow, and the computation is embarrassingly parallel. In this exercise, you will use Ray to execute the functions in parallel to speed it up.

The standard way to turn a Python function into a remote function is to add the `@ray.remote` decorator. Here is an example.

```python
# A regular Python function.
def regular_function(x):
    return x + 1

# A Ray remote function.
@ray.remote
def remote_function(x):
    return x + 1
```

The differences are the following:

1. **Invocation:** The regular version is called with `regular_function(1)`, whereas the remote version is called with `remote_function.remote(1)`.
2. **Return values:** `regular_function` immediately executes and returns `1`, whereas `remote_function` immediately returns an object ID (a future) and then creates a task that will be executed on a worker process. The result can be obtained with `ray.get`.
    ```python
    >>> regular_function(0)
    1
    
    >>> remote_function.remote(0)
    ObjectID(1c80d6937802cd7786ad25e50caf2f023c95e350)
    
    >>> ray.get(remote_function.remote(0))
    1
    ```
3. **Parallelism:** Invocations of `regular_function` happen **serially**, for example
    ```python
    # These happen serially.
    for _ in range(4):
        regular_function(0)
    ```
    whereas invocations of `remote_function` happen in **parallel**, for example
    ```python
    # These happen in parallel.
    for _ in range(4):
        remote_function.remote(0)
    ```

In [1]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import ray
import time
import numpy as np
from numpy import random
import os
import pickle

import sys

print("Python:", sys.version)
print("Ray:", ray.__version__)

Python: 3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:42:22) [GCC 14.3.0]
Ray: 2.55.1


Start Ray.

**Tier 1 / lab lokalnie (bez AWS):** w następnej komórce ustaw `USE_REMOTE_CLUSTER = False` - Ray uruchomi się **na tej maszynie** (w kontenerze Docker).

**Klaster z zajęć (opcjonalnie):** `USE_REMOTE_CLUSTER = True` i aktualny adres `ray://...` od prowadzącego.

Argument `ignore_reinit_error=True` pozwala ponownie uruchomić komórkę bez błędu.


In [2]:
STUDENT_ID = "kaliński_jakub_420841"  # imię_nazwisko_nr_albumu

# False = lokalny Ray (Docker / laptop). True = zdalny klaster (Ray Client, ray://...).
USE_REMOTE_CLUSTER = False
RAY_ADDRESS = "ray://HOST:PORT"  # tylko gdy USE_REMOTE_CLUSTER=True (adres od prowadzącego)

if ray.is_initialized():
    ray.shutdown()

if USE_REMOTE_CLUSTER:
    ray.init(
        address=RAY_ADDRESS,
        ignore_reinit_error=True,
        namespace=f"lab-ray-{STUDENT_ID}",
    )
    mode = "remote (Ray Client)"
else:
    ray.init(
        ignore_reinit_error=True,
        namespace=f"lab-ray-{STUDENT_ID}",
    )
    mode = "local"

print("Connected - mode:", mode)
print("Student:", STUDENT_ID)
print("Namespace:", ray.get_runtime_context().namespace)
print("Job ID:", ray.get_runtime_context().get_job_id())
print("Cluster resources:", ray.cluster_resources())


2026-05-31 23:27:50,711	WARNING services.py:2213 -- WARNING: The object store is using /tmp/ray instead of /dev/shm because /dev/shm has only 2147483648 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=2.44gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.
2026-05-31 23:28:02,320	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 
/home/ray/anaconda3/lib/python3.10/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Connected - mode: local
Student: kaliński_jakub_420841
Namespace: lab-ray-kaliński_jakub_420841
Job ID: 01000000
Cluster resources: {'CPU': 8.0, 'object_store_memory': 2376885043.0, 'node:172.19.0.2': 1.0, 'node:__internal_head__': 1.0, 'memory': 5546065101.0}


In [3]:
# This function is a proxy for a more interesting and computationally
# intensive function.

@ray.remote
def slow_function(i):
    print(f"Running in pid {os.getpid()}")
    time.sleep(1)
    return i

**EXERCISE:** The loop below takes too long. The four function calls could be executed in parallel. Instead of four seconds, it should only take one second. Once `slow_function` has been made a remote function, execute these four tasks in parallel by calling `slow_function.remote()`. Then obtain the results by calling `ray.get` on a list of the resulting object IDs.

In [4]:
# Sleep a little to improve the accuracy of the timing measurements below.
# We do this because workers may still be starting up in the background.
time.sleep(2.0)
start_time = time.time()

results = ray.get([slow_function.remote(i) for i in range(4)])

end_time = time.time()
duration = end_time - start_time

print('The results are {}. This took {} seconds. Run the next cell to see '
      'if the exercise was done correctly.'.format(results, duration))

(slow_function pid=670) Running in pid 670
The results are [0, 1, 2, 3]. This took 42.3324408531189 seconds. Run the next cell to see if the exercise was done correctly.


**VERIFY:** Run some checks to verify that the changes you made to the code were correct. Some of the checks should fail when you initially run the cells. After completing the exercises, the checks should pass.

In [5]:
assert results == [0, 1, 2, 3], 'Did you remember to call ray.get?'
assert duration < 3.1, ('The loop took {} seconds. This is too slow.'
                        .format(duration))
assert duration > 1, ('The loop took {} seconds. This is too fast.'
                      .format(duration))

print('Success! The example took {} seconds.'.format(duration))

AssertionError: The loop took 42.3324408531189 seconds. This is too slow.

***Task: Batch Processing***
Let's implement a simple batch image/data processing simulation. In the cell below, you have a function that processes many independent “log chunks”. Each chunk takes about 1 second.

In [ ]:
import time
import random
from collections import Counter

EVENT_TYPES = ["login", "logout", "purchase", "click", "error"]


def process_log_chunk(chunk_id, chunk_size=10_000):
    """
    Simulate processing one chunk of logs.
    """
    time.sleep(1)  # simulate I/O or expensive parsing

    events = [
        random.choice(EVENT_TYPES)
        for _ in range(chunk_size)
    ]

    counts = Counter(events)

    return {
        "chunk_id": chunk_id,
        "counts": dict(counts),
#        "hostname": socket.gethostname(), #uncomment the line for remote version
    }

Now execute it sequentially

In [ ]:
start = time.time()

sequential_results = [
    process_log_chunk(i)
    for i in range(8)
]

sequential_time = time.time() - start

print("Sequential time:", sequential_time)
sequential_results[:2]

Next implement task that will execute the processing logic on distributed cluster (name task as *process_log_chunk_remote*)

In [ ]:
@ray.remote
def process_log_chunk_remote(chunk_id, chunk_size=10_000):
    return process_log_chunk(chunk_id, chunk_size=chunk_size)

Execute the task, in parrallel way  - show the time difference in processing compared to the sequential version

In [ ]:
start = time.time()
parallel_refs = [process_log_chunk_remote.remote(i) for i in range(8)]
parallel_results = ray.get(parallel_refs)
parallel_time = time.time() - start

print("Parallel time:", parallel_time)
print("Sequential time (from earlier):", sequential_time)
print("Speedup:", sequential_time / parallel_time)
parallel_results[:2]

As example of an anti-pattern, execute tasks in sequential manner using the ray framework, print processing time

In [ ]:
start = time.time()
anti_results = []
for i in range(8):
    anti_results.append(ray.get(process_log_chunk_remote.remote(i)))
anti_time = time.time() - start

print("Sequential Ray (ray.get inside loop) time:", anti_time)
print("This is an anti-pattern: each task finishes before the next is submitted.")

Next, as an introduction to Actor, compute total counts on your local process

In [ ]:
total_counts = Counter()
for r in parallel_results:
    total_counts.update(r["counts"])
dict(total_counts)

Finally measure the time for different number of iteration 4,8,16,32 
* When does adding more tasks stop improving runtime?
* How many CPUs does Ray report?                       

In [ ]:
print("Total Ray CPUs:", ray.cluster_resources().get("CPU", 0))
print("Free Ray CPUs:", ray.available_resources().get("CPU", 0))

for n_chunks in (4, 8, 16, 32):
    t0 = time.time()
    ray.get([process_log_chunk_remote.remote(i) for i in range(n_chunks)])
    elapsed = time.time() - t0
    print(f"n_chunks={n_chunks} -> {elapsed:.3f}s")

***
## Part 2 - Parallel Data Processing with Task Dependencies

**GOAL:** The goal of this exercise is to show how to pass object IDs into remote functions to encode dependencies between tasks.

In this exercise, we construct a sequence of tasks each of which depends on the previous mimicking a data parallel application. Within each sequence, tasks are executed serially, but multiple sequences can be executed in parallel.

In this exercise, you will use Ray to parallelize the computation below and speed it up.

### Concept for this Exercise - Task Dependencies

Suppose we have two remote functions defined as follows.

```python
@ray.remote
def f(x):
    return x
```

Arguments can be passed into remote functions as usual.

```python
>>> x1_id = f.remote(1)
>>> ray.get(x1_id)
1

>>> x2_id = f.remote([1, 2, 3])
>>> ray.get(x2_id)
[1, 2, 3]
```

**Object IDs** can also be passed into remote functions. When the function actually gets executed, **the argument will be a retrieved as a regular Python object**.

```python
>>> y1_id = f.remote(x1_id)
>>> ray.get(y1_id)
1

>>> y2_id = f.remote(x2_id)
>>> ray.get(y2_id)
[1, 2, 3]
```

So when implementing a remote function, the function should expect a regular Python object regardless of whether the caller passes in a regular Python object or an object ID.

**Task dependencies affect scheduling.** In the example above, the task that creates `y1_id` depends on the task that creates `x1_id`. This has the following implications.

- The second task will not be executed until the first task has finished executing.
- If the two tasks are scheduled on different machines, the output of the first task (the value corresponding to `x1_id`) will be copied over the network to the machine where the second task is scheduled.


These are some helper functions that mimic an example pattern of a data parallel application.

**EXERCISE:** You will need to turn all of these functions into remote functions. When you turn these functions into remote function, you do not have to worry about whether the caller passes in an object ID or a regular object. In both cases, the arguments will be regular objects when the function executes. This means that even if you pass in an object ID, you **do not need to call `ray.get`** inside of these remote functions.

In [ ]:
@ray.remote
def load_data(filename):
    time.sleep(0.1)
    return np.ones((1000, 100))

@ray.remote
def normalize_data(data):
    time.sleep(0.1)
    return data - np.mean(data, axis=0)

@ray.remote
def extract_features(normalized_data):
    time.sleep(0.1)
    return np.hstack([normalized_data, normalized_data ** 2])

@ray.remote
def compute_loss(features):
    num_data, dim = features.shape
    time.sleep(0.1)
    return np.sum((np.dot(features, np.ones(dim)) - np.ones(num_data)) ** 2)

assert hasattr(load_data, 'remote'), 'load_data must be a remote function'
assert hasattr(normalize_data, 'remote'), 'normalize_data must be a remote function'
assert hasattr(extract_features, 'remote'), 'extract_features must be a remote function'
assert hasattr(compute_loss, 'remote'), 'compute_loss must be a remote function'

**EXERCISE:** The loop below takes too long. Parallelize the four passes through the loop by turning `load_data`, `normalize_data`, `extract_features`, and `compute_loss` into remote functions and then retrieving the losses with `ray.get`.

**NOTE:** You should only use **ONE** call to `ray.get`. For example, the object ID returned by `load_data` should be passed directly into `normalize_data` without needing to be retrieved by the driver.

In [ ]:
# Sleep a little to improve the accuracy of the timing measurements below.
time.sleep(2.0)
start_time = time.time()

losses = []
for filename in ['file1', 'file2', 'file3', 'file4']:
    data = load_data.remote(filename)
    normalized_data = normalize_data.remote(data)
    features = extract_features.remote(normalized_data)
    loss = compute_loss.remote(features)
    losses.append(loss)

losses = ray.get(losses)

print('The losses are {}.'.format(losses) + '\n')
loss = sum(losses)

end_time = time.time()
duration = end_time - start_time

print('The loss is {}. This took {} seconds. Run the next cell to see '
      'if the exercise was done correctly.'.format(loss, duration))

**VERIFY:** Run some checks to verify that the changes you made to the code were correct. Some of the checks should fail when you initially run the cells. After completing the exercises, the checks should pass.

In [ ]:
assert loss == 4000
assert duration < 0.8, ('The loop took {} seconds. This is too slow.'
                        .format(duration))
assert duration > 0.4, ('The loop took {} seconds. This is too fast.'
                        .format(duration))

print('Success! The example took {} seconds.'.format(duration))

***
## Part 3 - Introducing Actors

**Goal:** The goal of this exercise is to show how to create an actor and how to call actor methods.

See the documentation on actors at https://docs.ray.io/en/latest/ray-core/actors.html.

Sometimes you need a "worker" process to have "state". For example, that state might be a neural network, a simulator environment, a counter, or something else entirely. However, remote functions are side-effect free. That is, they operate on inputs and produce outputs, but they don't change the state of the worker they execute on.

Actors are different. When we instantiate an actor, a brand new worker is created, and all methods that are called on that actor are executed on the newly created worker.

This means that with a single actor, no parallelism can be achieved because calls to the actor's methods will be executed one at a time. However, multiple actors can be created and methods can be executed on them in parallel.

### Concepts for this Exercise - Actors

To create an actor, decorate Python class with the `@ray.remote` decorator.

```python
@ray.remote
class Example(object):
    def __init__(self, x):
        self.x = x
    
    def set(self, x):
        self.x = x
    
    def get(self):
        return self.x
```

Like regular Python classes, **actors encapsulate state that is shared across actor method invocations**.

Actor classes differ from regular Python classes in the following ways.
1. **Instantiation:** A regular class would be instantiated via `e = Example(1)`. Actors are instantiated via
    ```python
    e = Example.remote(1)
    ```
    When an actor is instantiated, a **new worker process** is created by a local scheduler somewhere in the cluster.
2. **Method Invocation:** Methods of a regular class would be invoked via `e.set(2)` or `e.get()`. Actor methods are invoked differently.
    ```python
    >>> e.set.remote(2)
    ObjectID(d966aa9b6486331dc2257522734a69ff603e5a1c)
    
    >>> e.get.remote()
    ObjectID(7c432c085864ed4c7c18cf112377a608676afbc3)
    ```
3. **Return Values:** Actor methods are non-blocking. They immediately return an object ID and **they create a task which is scheduled on the actor worker**. The result can be retrieved with `ray.get`.
    ```python
    >>> ray.get(e.set.remote(2))
    None
    
    >>> ray.get(e.get.remote())
    2
    ```

**EXERCISE:** Change the `Foo` class to be an actor class by using the `@ray.remote` decorator.

In [ ]:
@ray.remote
class Foo(object):
    def __init__(self):
        self.counter = 0

    def reset(self):
        self.counter = 0

    def increment(self):
        time.sleep(0.5)
        self.counter += 1
        return self.counter

assert hasattr(Foo, 'remote'), 'You need to turn "Foo" into an actor with @ray.remote.'

**EXERCISE:** Change the intantiations below to create two actors by calling `Foo.remote()`.

In [ ]:
# Create two Foo objects.
f1 = Foo.remote()
f2 = Foo.remote()

**EXERCISE:** Parallelize the code below. The two actors can execute methods in parallel (though each actor can only execute one method at a time).

In [ ]:
# Sleep a little to improve the accuracy of the timing measurements below.
time.sleep(2.0)
start_time = time.time()

# Reset the actor state so that we can run this cell multiple times without
# changing the results.
ray.get([f1.reset.remote(), f2.reset.remote()])

# We want to parallelize this code. However, it is not straightforward to
# make "increment" a remote function, because state is shared (the value of
# "self.counter") between subsequent calls to "increment". In this case, it
# makes sense to use actors.
refs = []
for _ in range(5):
    refs.append(f1.increment.remote())
    refs.append(f2.increment.remote())
results = ray.get(refs)

end_time = time.time()
duration = end_time - start_time

print('Success! The example took {} seconds.'.format(duration))

_ref_types = tuple(
    t for t in (getattr(ray, "ObjectRef", None), getattr(ray, "ObjectID", None)) if t is not None
)
assert not any(isinstance(result, _ref_types) for result in results), 'Looks like "results" is {}. You may have forgotten to call ray.get.'.format(results)

**VERIFY:** Run some checks to verify that the changes you made to the code were correct. Some of the checks should fail when you initially run the cells. After completing the exercises, the checks should pass.

In [ ]:
assert results == [1, 1, 2, 2, 3, 3, 4, 4, 5, 5]

assert duration < 3, ('The experiments ran in {} seconds. This is too '
                      'slow.'.format(duration))
assert duration > 2.5, ('The experiments ran in {} seconds. This is too '
                        'fast.'.format(duration))

print('Success! The example took {} seconds.'.format(duration))

**LogAggregator Task** Next, create an actor that will aggregate log events from the *Task Exercise* part of the class. The actor should implement methods such as: 
1. add_chunk_result - adding chunk
2. get_count - total counts
3. get_processed_chunks - current processed chunks
4. get_summary - return num_chunks, num_events and tital counts
5. reset - clears all store data


In [ ]:
from collections import Counter


@ray.remote
class LogAggregator:
    def __init__(self):
        self._counts = Counter()
        self._chunk_ids = []

    def add_chunk_result(self, result):
        self._chunk_ids.append(result["chunk_id"])
        self._counts.update(result["counts"])

    def get_count(self):
        return dict(self._counts)

    def get_processed_chunks(self):
        return list(self._chunk_ids)

    def get_summary(self):
        return {
            "num_chunks": len(self._chunk_ids),
            "num_events": sum(self._counts.values()),
            "total_counts": dict(self._counts),
        }

    def reset(self):
        self._counts.clear()
        self._chunk_ids.clear()

It will be named actor it means its name will be unique

In [ ]:
ACTOR_NAME = f"actor-{STUDENT_ID}"

try:
    aggregator = ray.get_actor(ACTOR_NAME)
    print(f"Using existing actor: {ACTOR_NAME}")
except ValueError:
    aggregator = LogAggregator.options(name=ACTOR_NAME).remote()
    print(f"Created new actor: {ACTOR_NAME}")

# Optional: reset state before exercise
ray.get(aggregator.reset.remote())

Next collect all data, and display summary

In [ ]:
refs = [process_log_chunk_remote.remote(i) for i in range(8)]
chunks = ray.get(refs)
for c in chunks:
    ray.get(aggregator.add_chunk_result.remote(c))

summary = ray.get(aggregator.get_summary.remote())
print(summary)

Next run more tasks and prove that the state is updated

In [ ]:
more_refs = [process_log_chunk_remote.remote(i) for i in range(8, 16)]
more_chunks = ray.get(more_refs)
for c in more_chunks:
    ray.get(aggregator.add_chunk_result.remote(c))

summary2 = ray.get(aggregator.get_summary.remote())
print("After second batch:", summary2)
print("Processed chunk ids (tail):", ray.get(aggregator.get_processed_chunks.remote())[-4:])

## Part 4 - More advanced Actors, queuing operations

**GOAL:** The goal of this exercise is to illustrate how to actors queues of different operations

### Concepts for this Exercise - learn different ways for serializing calls on actor 


As the base for the exercise, create the Actor that will implement bank account functionality. 
1. It should keep information regarding the owner, balance, and all operations.
2. The account should support deposit and withdrawal operations (both should have a 1-second sleep time for processing). On successful operations return set with owner, name of operation, amount, balance and hostname (socket.gethostname())
3. Add getters for  balance and history


In [ ]:
import socket


@ray.remote
class BankAccount:
    def __init__(self, owner, initial_balance=0):
        self.owner = owner
        self.balance = initial_balance
        self.history = []

    def deposit(self, amount):
        time.sleep(1)
        self.balance += amount
        rec = (self.owner, "deposit", amount, self.balance, socket.gethostname())
        self.history.append(rec)
        return rec

    def withdraw(self, amount):
        time.sleep(1)
        self.balance -= amount
        rec = (self.owner, "withdraw", amount, self.balance, socket.gethostname())
        self.history.append(rec)
        return rec

    def get_balance(self):
        return self.balance

    def get_history(self):
        return list(self.history)

The following operations should pass

In [ ]:
account = BankAccount.remote("kowalski", initial_balance=100)

start = time.time()

refs = [
    account.deposit.remote(10),
    account.deposit.remote(20),
    account.withdraw.remote(50),
    account.deposit.remote(5),
]

results = ray.get(refs)
duration = time.time() - start

print("Duration:", duration)
print("Results:")
for r in results:
    print(r)

print("Final balance:", ray.get(account.get_balance.remote()))
print("History:", ray.get(account.get_history.remote()))

1. How long does it take to execute the four operations?
2. Why does it take about 4 seconds, even though the calls were submitted almost at the same time?
3. Is the final account balance correct?
4. Does the actor behave more like a function or like a process with a message queue?


In [ ]:
print(
    "1) About 4 seconds (one second per method; the actor runs methods strictly in order).\n"
    "2) All four calls target the same actor, so they are queued and executed serially despite being submitted together.\n"
    "3) Yes: 100 + 10 + 20 - 50 + 5 = 85.\n"
    "4) Like a process with a message queue: requests are serialized on the actor worker."
)

Next create 4 different accounts and call single method on one of them, measure time

In [ ]:
accounts = [BankAccount.remote(f"user{i}", initial_balance=100) for i in range(4)]
start = time.time()
ray.get(accounts[0].deposit.remote(1))
elapsed = time.time() - start
print("Single deposit on one of four actors took:", elapsed, "seconds")

1. Why does it now take about 1 second instead of 4?
2. What is the unit of parallelism: the method or the actor?
3. What is the relationship between the number of actors and parallelism?


In [ ]:
print(
    "1) One second: only one actor does work; the other three never run a method.\n"
    "2) The actor is the unit of serialization; methods on different actors can overlap.\n"
    "3) More independent actors increase parallelism until you hit cluster CPU limits."
)

Finally lets create concurent version of the Bank account

In [ ]:
account2 = BankAccount.options(max_concurrency=4).remote("nowak", initial_balance=100)

start = time.time()

refs = [
    account2.deposit.remote(10),
    account2.deposit.remote(20),
    account2.withdraw.remote(50),
    account2.deposit.remote(5),
]

results = ray.get(refs)
duration = time.time() - start

print("Duration:", duration)
for r in results:
    print(r)

print("Final balance:", ray.get(account2.get_balance.remote()))
print("History:", ray.get(account2.get_history.remote()))

1. Was the execution faster?
2. Is the final state always correct?
3. Why can concurrency inside an actor be dangerous?
4. When is it better to use multiple actors, and when is it better to use a single actor with `max_concurrency`?
(https://docs.ray.io/en/latest/ray-core/api/doc/ray.actor.ActorClass.options.html)

In [ ]:
print(
    "1) Yes, wall time drops toward one second because up to four methods can run concurrently on one actor.\n"
    "2) No: concurrent updates to shared balance/history can race unless you add locking or atomic updates.\n"
    "3) Shared mutable state without synchronization leads to lost updates and inconsistent history.\n"
    "4) Prefer multiple actors when natural partitions exist (separate accounts). Use max_concurrency only when methods are independent or externally synchronized (I/O-bound pipelines)."
)

***
## Part 5 - Handling Slow Tasks

**GOAL:** The goal of this exercise is to show how to use `ray.wait` to avoid waiting for slow tasks.

See the documentation for ray.wait at https://docs.ray.io/en/latest/ray-core/api/doc/ray.wait.html.

This script starts 6 tasks, each of which takes a random amount of time to complete. We'd like to process the results in two batches (each of size 3). Change the code so that instead of waiting for a fixed set of 3 tasks to finish, we make the first batch consist of the first 3 tasks that complete. The second batch should consist of the 3 remaining tasks. Do this exercise by using `ray.wait`.

### Concepts for this Exercise - ray.wait

After launching a number of tasks, you may want to know which ones have finished executing. This can be done with `ray.wait`. The function works as follows.

```python
ready_ids, remaining_ids = ray.wait(object_ids, num_returns=1, timeout=None)
```

**Arguments:**
- `object_ids`: This is a list of object IDs.
- `num_returns`: This is maximum number of object IDs to wait for. The default value is `1`.
- `timeout`: This is the maximum amount of time in milliseconds to wait for. So `ray.wait` will block until either `num_returns` objects are ready or until `timeout` milliseconds have passed.

**Return values:**
- `ready_ids`: This is a list of object IDs that are available in the object store.
- `remaining_ids`: This is a list of the IDs that were in `object_ids` but are not in `ready_ids`, so the IDs in `ready_ids` and `remaining_ids` together make up all the IDs in `object_ids`.

Define a remote function that takes a variable amount of time to run.

In [ ]:
@ray.remote
def f(i):
    np.random.seed(5 + i)
    x = np.random.uniform(0, 4)
    time.sleep(x)
    return i, time.time()

**EXERCISE:** Using `ray.wait`, change the code below so that `initial_results` consists of the outputs of the first three tasks to complete instead of the first three tasks that were submitted.

In [ ]:
# Sleep a little to improve the accuracy of the timing measurements below.
time.sleep(2.0)
start_time = time.time()

# This launches 6 tasks, each of which takes a random amount of time to
# complete.
result_ids = [f.remote(i) for i in range(6)]
# First batch: the three tasks that finish first (not the first three submitted).
ready_ids, pending_ids = ray.wait(result_ids, num_returns=3)
initial_results = ray.get(ready_ids)

end_time = time.time()
duration = end_time - start_time

**EXERCISE:** Change the code below so that `remaining_results` consists of the outputs of the last three tasks to complete.

In [ ]:
# Wait for the remaining tasks to complete.
remaining_results = ray.get(pending_ids)

**VERIFY:** Run some checks to verify that the changes you made to the code were correct. Some of the checks should fail when you initially run the cells. After completing the exercises, the checks should pass.

In [ ]:
assert len(initial_results) == 3
assert len(remaining_results) == 3

initial_indices = [result[0] for result in initial_results]
initial_times = [result[1] for result in initial_results]
remaining_indices = [result[0] for result in remaining_results]
remaining_times = [result[1] for result in remaining_results]

assert set(initial_indices + remaining_indices) == set(range(6))

assert duration < 1.5, ('The initial batch of ten tasks was retrieved in '
                        '{} seconds. This is too slow.'.format(duration))

assert duration > 0.8, ('The initial batch of ten tasks was retrieved in '
                        '{} seconds. This is too fast.'.format(duration))

# Make sure the initial results actually completed first.
assert max(initial_times) < min(remaining_times)

print('Success! The example took {} seconds.'.format(duration))

## Part 6 - Speed up Serialization

**GOAL:** The goal of this exercise is to illustrate how to speed up serialization by using `ray.put`.

### Concepts for this Exercise - ray.put

Object IDs can be created in multiple ways.
- They are returned by remote function calls.
- They are returned by actor method calls.
- They are returned by `ray.put`.

When an object is passed to `ray.put`, the object is serialized using the Apache Arrow format (see https://arrow.apache.org/ for more information about Arrow) and copied into a shared memory object store. This object will then be available to other workers on the same machine via shared memory. If it is needed by workers on another machine, it will be shipped under the hood.

**When objects are passed into a remote function, Ray puts them in the object store under the hood.** That is, if `f` is a remote function, the code

```python
x = np.zeros(1000)
f.remote(x)
```

is essentially transformed under the hood to

```python
x = np.zeros(1000)
x_id = ray.put(x)
f.remote(x_id)
```

The call to `ray.put` copies the numpy array into the shared-memory object store, from where it can be read by all of the worker processes (without additional copying). However, if you do something like

```python
for i in range(10):
    f.remote(x)
```

then 10 copies of the array will be placed into the object store. This takes up more memory in the object store than is necessary, and it also takes time to copy the array into the object store over and over. This can be made more efficient by placing the array in the object store only once as follows.

```python
x_id = ray.put(x)
for i in range(10):
    f.remote(x_id)
```

In this exercise, you will speed up the code below and reduce the memory footprint by calling `ray.put` on the neural net weights before passing them into the remote functions.

**WARNING:** This exercise requires a lot of memory to run. If this notebook is running within a Docker container, then the docker container must be started with a large shared-memory file system. This can be done by starting the docker container with the `--shm-size` flag.

In [ ]:
neural_net_weights = {'variable{}'.format(i): np.random.normal(size=1000000)
                      for i in range(50)}

**EXERCISE:** Compare the time required to serialize the neural net weights and copy them into the object store using Ray versus the time required to pickle and unpickle the weights. The big win should be with the time required for *deserialization*.

Note that when you call `ray.put`, in addition to serializing the object, we are copying it into shared memory where it can be efficiently accessed by other workers on the same machine.

**NOTE:** You don't actually have to do anything here other than run the cell below and read the output.

**NOTE:** Sometimes `ray.put` can be faster than `pickle.dumps`. This is because `ray.put` leverages multiple threads when serializing large objects. Note that this is not possible with `pickle`.

In [ ]:
print('Ray - serializing')
%time x_id = ray.put(neural_net_weights)
print('\nRay - deserializing')
%time x_val = ray.get(x_id)

print('\npickle - serializing')
%time serialized = pickle.dumps(neural_net_weights)
print('\npickle - deserializing')
%time deserialized = pickle.loads(serialized)

Define a remote function which uses the neural net weights.

In [ ]:
@ray.remote
def use_weights(weights, i):
    len(weights)
    return i

**EXERCISE:** In the code below, use `ray.put` to avoid copying the neural net weights to the object store multiple times.

In [ ]:
# Sleep a little to improve the accuracy of the timing measurements below.
time.sleep(2.0)
start_time = time.time()

weights_id = ray.put(neural_net_weights)
results = ray.get([use_weights.remote(weights_id, i) for i in range(20)])

end_time = time.time()
duration = end_time - start_time

**VERIFY:** Run some checks to verify that the changes you made to the code were correct. Some of the checks should fail when you initially run the cells. After completing the exercises, the checks should pass.

In [ ]:
assert results == list(range(20))
assert duration < 1, ('The experiments ran in {} seconds. This is too '
                      'slow.'.format(duration))

print('Success! The example took {} seconds.'.format(duration))

In [ ]:
# Example: Parameter Server distributed application with Ray Actors
# Problem: We want to update weights and gradients, computed by workers, at a central server.
# Let's use Python class and convert that to a remote Actor class actor as a Parameter Server.
# This is a common example in machine learning where you have a central
# Parameter server updating gradients from other worker processes computing individual gradients.

print('parameter server')
@ray.remote
class ParameterSever:
    def __init__(self):
        # Initialized our gradients to zero
        self.params = np.zeros(10)

    def get_params(self):
        # Return current gradients
        return self.params

    def update_params(self, grad):
        # Update the gradients
        self.params -= grad

# Define a worker or task as a function for a remote Worker. This could be a
# machine learning function that computes gradients and sends them to
# the parameter server.

@ray.remote
def worker(ps):         # It takes an actor handle or instance as an argument
    time.sleep(2)
    params = ray.get(ps.get_params.remote())
    grad = np.random.normal(size=params.shape)
    ray.get(ps.update_params.remote(grad))


# Start our Parameter Server actor. This will be scheduled as a worker process
# on a remote Ray node. You invoke its ActorClass.remote(...) to instantiate an
# Actor instance of that type.

param_server = ParameterSever.remote()
print(param_server)

# Let's get the initial values of the parameter server
print(f"Initial params: {ray.get(param_server.get_params.remote())}")

# Create Workers remote tasks computing gradients
# Let's create three separate worker tasks as our machine learning tasks
# that compute gradients. These will be scheduled as tasks on a Ray cluster.

# You can use list comprehension.
# If we need more workers to scale, we can always bump them up.
# Note: We are sending the parameter_server as an argument to the remote worker task.

print([worker.remote(param_server) for _ in range(3)])

# Now, let's iterate over a loop and query the Parameter Server as the
# workers are running independently and updating the gradients

for _i in range(20):
    print(f"Updated params: {ray.get(param_server.get_params.remote())}")
    time.sleep(1)

---
# Zadanie domowe

**Wymagania (ray-zadanie.pdf):** Jupyter z (1) obliczeniem π metodą Monte Carlo z aktorem nadzorującym, (2) równoległym merge sort + porównanie z wersją sekwencyjną.

**Architektura:**
- **π:** `PiSupervisor` (aktor, stan globalny) + `mc_batch` (remote tasks równolegle liczą próbki w kwadracie [0,1]²).
- **Sortowanie:** `merge_sort_sequential` vs `merge_sort_remote` - rekurencyjny podział; poniżej progu `REMOTE_THRESHOLD` sort lokalny (mniej overheadu Ray).

## 7.1 Równoległe sortowanie przez scalanie


In [ ]:
def merge(left, right):
    """Scalanie dwóch posortowanych list."""
    out = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            out.append(left[i])
            i += 1
        else:
            out.append(right[j])
            j += 1
    out.extend(left[i:])
    out.extend(right[j:])
    return out


def merge_sort_sequential(arr):
    """Wersja referencyjna - cała praca na driverze."""
    if len(arr) <= 1:
        return list(arr)
    mid = len(arr) // 2
    return merge(
        merge_sort_sequential(arr[:mid]),
        merge_sort_sequential(arr[mid:]),
    )


# Poniżej tego rozmiaru nie spawnujemy kolejnych tasków Ray (koszt serializacji > zysk).
REMOTE_THRESHOLD = 8_000


@ray.remote
def merge_sort_remote(arr):
    if len(arr) <= REMOTE_THRESHOLD:
        return merge_sort_sequential(arr)
    mid = len(arr) // 2
    left_ref = merge_sort_remote.remote(arr[:mid])
    right_ref = merge_sort_remote.remote(arr[mid:])
    left, right = ray.get([left_ref, right_ref])
    return merge(left, right)


SIZE = 50_000
rng = np.random.default_rng(42)
data = rng.integers(0, 1_000_000, size=SIZE).tolist()

print(f"Sorting {SIZE} integers (REMOTE_THRESHOLD={REMOTE_THRESHOLD})\n")

t0 = time.time()
sorted_seq = merge_sort_sequential(data)
t_seq = time.time() - t0
print(f"Sequential merge sort: {t_seq:.3f} s")

t0 = time.time()
sorted_par = ray.get(merge_sort_remote.remote(data))
t_par = time.time() - t0
print(f"Parallel merge sort:     {t_par:.3f} s")

assert sorted_seq == sorted_par == sorted(data)
print(f"Correctness OK. Speedup (seq/par): {t_seq / t_par:.2f}x")


## 7.2 Podsumowanie Tier 1 - sortowanie

Wersja równoległa dzieli tablicę rekurencyjnie; lewa i prawa połówka sortowane są **równocześnie** jako osobne taski Ray. Scalanie odbywa się po `ray.get` na obu gałęziach. Próg `REMOTE_THRESHOLD` ogranicza głębokość drzewa tasków.

---
## 8. Obliczanie π - Monte Carlo (aktor + remote tasks)

Losujemy punkty w kwadracie [0,1]×[0,1]. Ułamek punktów w ćwiartce koła (x²+y²≤1) × 4 ≈ π.

![monte-carlo](img/monte_carlo_pi_sma.png)

Oczekiwany format wyjścia:
```
Current value of π is: 3.141...
```


In [ ]:
NUM_WORKERS = 8
SAMPLES_PER_WORKER = 200_000
ROUNDS = 16


@ray.remote
def mc_batch(num_samples):
    """Jeden worker - liczy punkty w ćwiartce koła jednostkowego."""
    rng = np.random.default_rng()
    x = rng.random(num_samples)
    y = rng.random(num_samples)
    inside = int(np.count_nonzero(x * x + y * y <= 1.0))
    return inside, num_samples


@ray.remote
class PiSupervisor:
    """Aktor przechowujący skumulowane trafienia - stan żyje między rundami."""

    def __init__(self):
        self.inside = 0
        self.total = 0

    def run_round(self, num_workers, samples_per_worker):
        refs = [mc_batch.remote(samples_per_worker) for _ in range(num_workers)]
        for inside, n in ray.get(refs):
            self.inside += inside
            self.total += n
        return 4.0 * self.inside / self.total


supervisor = PiSupervisor.remote()
print(f"Monte Carlo π: {NUM_WORKERS} workers × {SAMPLES_PER_WORKER} samples, {ROUNDS} rounds\n")

for r in range(ROUNDS):
    pi = ray.get(supervisor.run_round.remote(NUM_WORKERS, SAMPLES_PER_WORKER))
    print(f"Current value of π is: {pi}")

print("\n--- Tier 1 (π) gotowe ---")
print(f"Końcowa estymata: {pi:.15f}  (błąd względem math.pi: {abs(pi - np.pi):.2e})")


## Clean up  - Clean up the environemnt

**GOAL:** The goal of this exercise is to close the environment once you finish play with ray `ray.shutdown`.

In [ ]:
ray.shutdown()
print("Ray shutdown.")
